# DIU Konnect: Standalone Agentic RAG Pipeline

This notebook demonstrates an end-to-end Agentic RAG pipeline for Daffodil International University (DIU).

**Features:**
- Scrapes official DIU web pages (Noticeboard, Alumni Guidelines, Alumni Portal, Scholarships)
- Handles Next.js client-side dynamic rendering to capture full accordion policies and notice attachments/PDFs from DIU's backend APIs
- Indexes text chunks using `text-embedding-3-small` into a local Chroma vector database
- Uses a LangChain / LangGraph ReAct agent powered by `gpt-4o-mini`
- Formulates canonical retrieval queries across English, Bengali, and Banglish, and delivers polite, professional English responses with exact source citations

### 1. Install & Import Dependencies

In [ ]:
# In a fresh environment, uncomment and run:
# !pip install -q langchain langchain-openai langchain-chroma langchain-community langgraph chromadb beautifulsoup4 requests python-dotenv pypdf

import os
import io
import logging
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

try:
    import pypdf
    logging.getLogger("pypdf").setLevel(logging.ERROR)
except ImportError:
    pypdf = None

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.tools import tool

try:
    from langchain.agents import create_agent
except ImportError:
    create_agent = None

try:
    from langgraph.prebuilt import create_react_agent
except ImportError:
    create_react_agent = None

load_dotenv()

### 2. Configure OpenAI API Key

In [ ]:
if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = "your-api-key-here"

print("OpenAI API Key configured:", bool(os.getenv("OPENAI_API_KEY")))

### 3. Scrape Official DIU Target URLs (Including Dynamic Accordions and Notice PDFs)

In [ ]:
TARGET_URLS = [
    "https://daffodilvarsity.edu.bd/noticeboard",
    "https://alumni.daffodilvarsity.edu.bd/articles/membership-guidelines-28",  # Direct alumni card guideline
    "https://alumni.daffodilvarsity.edu.bd/",
    "https://daffodilvarsity.edu.bd/scholarship/diu-scholarship"
]

def fetch_dynamic_scholarships(source_url: str, headers: dict):
    docs = []
    api_url = "https://webbackend.daffodilvarsity.edu.bd/api/v2/public/accordion/scholarship"
    try:
        r = requests.get(api_url, headers=headers, timeout=15)
        if r.status_code == 200:
            data = r.json().get("data", {})
            for cat in data.get("categories", []):
                cat_title = cat.get("title", "")
                for acc in cat.get("accordions", []):
                    acc_title = acc.get("title", "")
                    desc = acc.get("description", "")
                    soup = BeautifulSoup(desc, "html.parser")
                    for t in soup.find_all("table"):
                        rows = []
                        for tr in t.find_all("tr"):
                            cells = [td.get_text(" ", strip=True) for td in tr.find_all(["th", "td"])]
                            if cells:
                                rows.append(" | ".join(cells))
                        t.replace_with("\n" + "\n".join(rows) + "\n")
                    content = (
                        f"DIU Scholarship & Waiver Policy\n"
                        f"Category: {cat_title}\n"
                        f"Title: {acc_title}\n\n"
                        f"{soup.get_text(separator='\n', strip=True)}"
                    )
                    docs.append(Document(page_content=content, metadata={"source": source_url}))
            print(f"  ✓ Extracted {len(docs)} dynamic scholarship & waiver policies from DIU backend.")
    except Exception as e:
        print(f"  ✗ Error fetching dynamic scholarship content: {e}")
    return docs

def fetch_dynamic_notices(source_url: str, headers: dict):
    docs = []
    api_url = "https://webbackend.daffodilvarsity.edu.bd/api/v1/public/notice?per_page=40"
    base_notice_url = source_url.rstrip("/")
    try:
        r = requests.get(api_url, headers=headers, timeout=15)
        if r.status_code == 200:
            notices = r.json().get("data", [])
            for n in notices:
                title = n.get("title", "").strip()
                slug = n.get("slug", "").strip()
                notice_url = f"{base_notice_url}/{slug}" if slug else base_notice_url
                category = n.get("notice_category", "")
                dept = n.get("department", "")
                date = n.get("create_at") or n.get("date") or ""
                pdf_text = ""
                file_urls = []
                for f in n.get("noticeFiles", []):
                    fname = f.get("file_name", "")
                    if fname:
                        file_urls.append(fname)
                        if pypdf is not None and fname.lower().endswith(".pdf"):
                            try:
                                pr = requests.get(fname, headers=headers, timeout=10)
                                if pr.status_code == 200:
                                    reader = pypdf.PdfReader(io.BytesIO(pr.content))
                                    pages_text = []
                                    for p in reader.pages[:4]:
                                        t = p.extract_text()
                                        if t:
                                            pages_text.append(t.strip())
                                    if pages_text:
                                        pdf_text += "\n[Attached Notice PDF Content]:\n" + "\n".join(pages_text)
                            except Exception:
                                pass
                content = (
                    f"DIU Official Notice\n"
                    f"Title: {title}\n"
                    f"Department: {dept}\n"
                    f"Category: {category}\n"
                    f"Published Date: {date}\n"
                    f"Notice URL: {notice_url}\n"
                )
                if file_urls:
                    content += f"Attachments: {', '.join(file_urls)}\n"
                if pdf_text:
                    content += pdf_text
                docs.append(Document(page_content=content, metadata={"source": notice_url}))
            print(f"  ✓ Extracted {len(docs)} dynamic notices (with PDF contents) from DIU backend.")
    except Exception as e:
        print(f"  ✗ Error fetching dynamic noticeboard content: {e}")
    return docs

def scrape_pages(urls):
    documents = []
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    for url in urls:
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code == 200:
                soup = BeautifulSoup(resp.text, "html.parser")
                for element in soup(["script", "style", "nav", "footer", "header", "noscript"]):
                    element.decompose()
                text = " ".join(soup.stripped_strings)
                documents.append(Document(page_content=text, metadata={"source": url}))
                print(f"✓ Scraped: {url} ({len(text)} chars)")
            else:
                print(f"✗ Failed to fetch {url}: HTTP {resp.status_code}")
        except Exception as e:
            print(f"✗ Error fetching {url}: {e}")
        if "scholarship" in url:
            documents.extend(fetch_dynamic_scholarships(url, headers))
        elif "noticeboard" in url:
            documents.extend(fetch_dynamic_notices(url, headers))
    return documents

raw_docs = scrape_pages(TARGET_URLS)

### 4. Split Chunks and Index in Vector Store

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
chunks = text_splitter.split_documents(raw_docs)
print(f"Total chunks indexed: {len(chunks)}")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="diu_knowledge_base"
)
print("Vector store successfully built!")

### 5. Define Knowledge Base Retrieval Tool

In [ ]:
@tool
def search_diu_knowledge_base(query: str) -> str:
    """
    Search exclusively within Daffodil International University (DIU) 
    official sources: Alumni affairs, Noticeboard, and Scholarships.
    """
    results = vector_store.similarity_search(query, k=4)
    if not results:
        return "No relevant records found in the verified DIU sources."
    
    formatted = []
    for doc in results:
        formatted.append(f"[Source: {doc.metadata['source']}]\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)

### 6. Create Agent with Professional & Polite English System Instructions

In [ ]:
system_prompt = (
    "You are KonnectBuddy, a professional and courteous AI assistant dedicated to DIU Konnect. "
    "Your knowledge base is STRICTLY RESTRICTED to DIU Noticeboard, Alumni, and Scholarship pages. "
    "\n\nRules:"
    "\n1. Rely solely on the retrieved tool output. Do not make assumptions or fabricate info."
    "\n2. Always provide your final response in clear, professional English maintaining a polite and helpful tone, regardless of whether the user query is in English, Bengali, or Banglish."
    "\n3. Formulate your internal tool search queries in precise, canonical English terms."
    "\n4. Present details clearly and elegantly using structured lists, markdown tables, or bullet points where appropriate."
    "\n5. Always cite the exact source URL provided in the retrieved context."
    "\n6. If the requested information is absent from the official sources, politely inform the user that it is not available in the official records and offer guidance on where they may inquire further."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

if create_agent is not None:
    agent = create_agent(model=llm, tools=[search_diu_knowledge_base], system_prompt=system_prompt)
elif create_react_agent is not None:
    try:
        agent = create_react_agent(model=llm, tools=[search_diu_knowledge_base], prompt=system_prompt)
    except TypeError:
        agent = create_react_agent(model=llm, tools=[search_diu_knowledge_base], state_modifier=system_prompt)
else:
    raise ImportError("Agent factory not found.")

def ask_konnectbuddy(user_query: str):
    print(f"\nUser Query: {user_query}")
    print("-" * 50)
    response = agent.invoke({"messages": [("user", user_query)]})
    final_reply = response["messages"][-1].content
    print(final_reply)
    return final_reply

### 7. Evaluation & Test Queries

In [ ]:
# Query 1: Course offer notice
ask_konnectbuddy("Course Offer of Fall 2026")

In [ ]:
# Query 2: Banglish query (Alumni card)
ask_konnectbuddy("@konnectbuddy kivabe alumni card collect korte hobe?")

In [ ]:
# Query 3: SSC & HSC A+ Waiver
ask_konnectbuddy("amr hsc and ss te a+ ache ,ami kototuku waiver pabo")